# Qiskit Fall Fest 2026 — Phase 1, PS-O2: Portfolio Optimization

This notebook formulates the four-asset portfolio problem as a QUBO, verifies the encoding against exhaustive classical enumeration, evaluates a depth-one QAOA circuit, and compares the reference Processor A and Processor B coupling maps using the prescribed transpilation settings and synthetic noise model.

**Reproducibility:** run all cells from top to bottom in a fresh Google Colab runtime. The notebook saves its comparison table, shot-level summary, JSON metrics, and figures under `results/`.

**Parameter transparency:** the problem guide specifies the returns, covariance matrix, and budget constraint, but it does not prescribe numerical values for risk-aversion `q` or the QUBO penalty. This experiment uses `q = 2.0` and penalty `λ = 20.0` as documented experimental choices. Results are conditional on these choices and on the stated depth-one QAOA ansatz.

## 1. Environment and imports

The pinned versions below are intended for the Colab notebook runtime.

In [ ]:
%pip install -q "qiskit==2.5.2" "qiskit-aer==0.17.2" scipy pandas matplotlib

In [ ]:
import sys
import os
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import qiskit
import qiskit_aer
import matplotlib.pyplot as plt

from scipy.optimize import differential_evolution
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.quantum_info import Statevector
from qiskit.transpiler import CouplingMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

SEED = 42
SHOTS = 4096
RESULTS_DIR = Path("results")
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Qiskit:", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)
print("Shots per processor:", SHOTS)

## 2. Official PS-O2 inputs

The returns, covariance matrix, and cardinality constraint below are taken from the Phase 1 problem guide.

In [ ]:
NUM_ASSETS = 4
RETURNS = np.array([0.12, 0.18, 0.10, 0.22], dtype=float)
BUDGET = 2
SIGMA = np.array([
    [0.04, 0.01, 0.02, 0.01],
    [0.01, 0.09, 0.01, 0.03],
    [0.02, 0.01, 0.05, 0.02],
    [0.01, 0.03, 0.02, 0.08],
], dtype=float)

assert RETURNS.shape == (NUM_ASSETS,)
assert SIGMA.shape == (NUM_ASSETS, NUM_ASSETS)
assert np.allclose(SIGMA, SIGMA.T)
assert np.all(np.isfinite(RETURNS)) and np.all(np.isfinite(SIGMA))
assert BUDGET == 2

print("Expected returns:", RETURNS.tolist())
print("Budget (assets selected):", BUDGET)
print("Covariance matrix:\n", SIGMA)
print("Covariance eigenvalues:", np.linalg.eigvalsh(SIGMA))

## 3. Objective and feasible portfolios

We use the guide’s objective form `wᵀΣw − q μᵀw` subject to `sum(wᵢ) = k`. Since the numerical risk-aversion value and penalty are not specified in the guide, the values below are experimental choices. The penalty converts the equality constraint into an unconstrained binary objective.

In [ ]:
RISK_AVERSION = 2.0  # Experimental choice; not specified numerically in the guide
PENALTY = 20.0       # Experimental choice; not specified numerically in the guide
QAOA_P = 1


def direct_objective(w):
    """Binary objective with a squared cardinality-constraint penalty."""
    w = np.asarray(w, dtype=float)
    return float(w @ SIGMA @ w - RISK_AVERSION * (RETURNS @ w)
                 + PENALTY * (w.sum() - BUDGET) ** 2)

all_bits = list(itertools.product([0, 1], repeat=NUM_ASSETS))
feasible_portfolios = []
for bits in all_bits:
    x = np.asarray(bits, dtype=float)
    if int(x.sum()) == BUDGET:
        feasible_portfolios.append({
            "bits": list(map(int, bits)),
            "assets_zero_indexed": [i for i, bit in enumerate(bits) if bit],
            "return": float(RETURNS @ x),
            "risk": float(x @ SIGMA @ x),
            "objective": direct_objective(x),
        })

feasible_df = pd.DataFrame(feasible_portfolios).sort_values("objective")
display(feasible_df)
assert len(feasible_df) == 6
print("PASS: exactly six portfolios satisfy the two-asset constraint.")

## 4. QUBO encoding and verification

We use the upper-triangular convention `E(x) = offset + Σᵢ Qᵢᵢxᵢ + Σᵢ<ⱼ Qᵢⱼxᵢxⱼ`. The offset is retained for reporting, although a constant offset does not affect QAOA measurement probabilities.

In [ ]:
Q = np.zeros((NUM_ASSETS, NUM_ASSETS), dtype=float)
for i in range(NUM_ASSETS):
    Q[i, i] = SIGMA[i, i] - RISK_AVERSION * RETURNS[i] + PENALTY * (1 - 2 * BUDGET)
    for j in range(i + 1, NUM_ASSETS):
        Q[i, j] = 2 * SIGMA[i, j] + 2 * PENALTY
Q_OFFSET = PENALTY * BUDGET**2


def qubo_energy(bits):
    x = np.asarray(bits, dtype=float)
    value = Q_OFFSET
    for i in range(NUM_ASSETS):
        value += Q[i, i] * x[i]
        for j in range(i + 1, NUM_ASSETS):
            value += Q[i, j] * x[i] * x[j]
    return float(value)

qubo_errors = [abs(qubo_energy(bits) - direct_objective(bits)) for bits in all_bits]
print("QUBO matrix:\n", Q)
print("Constant offset:", Q_OFFSET)
print("Maximum absolute difference over all 16 bitstrings:", max(qubo_errors))
assert max(qubo_errors) < 1e-9
print("PASS: QUBO energy matches the direct objective for all 16 bitstrings.")

## 5. Exact classical baseline

All 16 binary strings are enumerated. The exact solution is used as the reference for evaluating QAOA; it is not a quantum result.

In [ ]:
classical_results = []
for bits in all_bits:
    x = np.asarray(bits, dtype=float)
    feasible = int(x.sum()) == BUDGET
    classical_results.append({
        "bits": list(map(int, bits)),
        "assets_zero_indexed": [i for i, bit in enumerate(bits) if bit],
        "selected_count": int(x.sum()),
        "return": float(RETURNS @ x),
        "risk": float(x @ SIGMA @ x),
        "objective": direct_objective(bits),
        "qubo_energy": qubo_energy(bits),
        "feasible": feasible,
    })

classical_df = pd.DataFrame(classical_results)
classical_optimum = min(classical_results, key=lambda row: row["objective"])
best_feasible = min((row for row in classical_results if row["feasible"]),
                    key=lambda row: row["objective"])
display(classical_df.sort_values("objective").head(8))
print("Exact penalized optimum:", classical_optimum)
print("Best feasible portfolio:", best_feasible)
assert classical_optimum["feasible"]
assert classical_optimum["bits"] == best_feasible["bits"]
print("PASS: the exact penalized optimum is feasible and agrees with the best feasible portfolio.")

## 6. Build the depth-one QAOA circuit

The cost unitary is derived from the upper-triangular QUBO convention above. This notebook uses a standard transverse-field mixer and one QAOA layer (`p = 1`).

In [ ]:
def build_qaoa_circuit(Q_matrix):
    Q_matrix = np.asarray(Q_matrix, dtype=float)
    if Q_matrix.shape != (NUM_ASSETS, NUM_ASSETS):
        raise ValueError("Q matrix must be 4x4.")
    if not np.all(np.isfinite(Q_matrix)):
        raise ValueError("Q matrix must contain finite values.")

    gamma = Parameter("gamma")
    beta = Parameter("beta")
    qc = QuantumCircuit(NUM_ASSETS)
    qc.h(range(NUM_ASSETS))

    # Convert x_i = (1 - Z_i)/2 into a cost Hamiltonian.
    z_coeff = np.zeros(NUM_ASSETS, dtype=float)
    for i in range(NUM_ASSETS):
        z_coeff[i] = -Q_matrix[i, i] / 2
    for i in range(NUM_ASSETS):
        for j in range(i + 1, NUM_ASSETS):
            coeff = Q_matrix[i, j]
            z_coeff[i] -= coeff / 4
            z_coeff[j] -= coeff / 4
            if not np.isclose(coeff, 0.0):
                qc.cx(i, j)
                qc.rz(gamma * coeff / 2, j)
                qc.cx(i, j)
    for i in range(NUM_ASSETS):
        if not np.isclose(z_coeff[i], 0.0):
            qc.rz(2 * gamma * z_coeff[i], i)
    for i in range(NUM_ASSETS):
        qc.rx(2 * beta, i)
    qc.measure_all()
    return qc

circuit = build_qaoa_circuit(Q)
print("Logical qubits:", circuit.num_qubits)
print("QAOA depth p:", QAOA_P)
print("Parameters:", sorted(p.name for p in circuit.parameters))
print("Logical circuit depth:", circuit.depth())
print("Logical operations:", dict(circuit.count_ops()))

## 7. Ideal statevector optimization

This parameter search is an ideal, noiseless simulation. It is not a Processor A/B result. Differential Evolution is seeded for reproducibility; a successful optimizer termination does not guarantee a global optimum.

In [ ]:
def expected_qubo_cost(params):
    gamma_value, beta_value = params
    parameter_map = {p.name: p for p in circuit.parameters}
    bound = circuit.assign_parameters({
        parameter_map["gamma"]: float(gamma_value),
        parameter_map["beta"]: float(beta_value),
    })
    unitary = bound.remove_final_measurements(inplace=False)
    probs = Statevector.from_instruction(unitary).probabilities()
    return float(sum(
        prob * qubo_energy([(index >> i) & 1 for i in range(NUM_ASSETS)])
        for index, prob in enumerate(probs)
    ))

optimizer_result = differential_evolution(
    expected_qubo_cost,
    bounds=[(0.0, 0.2), (0.0, np.pi)],
    seed=SEED,
    maxiter=100,
    popsize=12,
    tol=1e-7,
    polish=True,
    workers=1,
)
GAMMA_BEST = float(optimizer_result.x[0])
BETA_BEST = float(optimizer_result.x[1])
IDEAL_EXPECTED_ENERGY = float(optimizer_result.fun)
print("Best gamma:", GAMMA_BEST)
print("Best beta:", BETA_BEST)
print("Ideal expected energy:", IDEAL_EXPECTED_ENERGY)
print("Optimizer success:", bool(optimizer_result.success))
print("Optimizer message:", optimizer_result.message)
print("Exact classical optimum energy:", classical_optimum["objective"])
print("Absolute energy gap:", abs(IDEAL_EXPECTED_ENERGY - classical_optimum["objective"]))

## 8. Ideal output distribution and baseline quality

These probabilities come from the ideal statevector, not from finite-shot noisy simulation.

In [ ]:
parameter_map = {p.name: p for p in circuit.parameters}
bound = circuit.assign_parameters({
    parameter_map["gamma"]: GAMMA_BEST,
    parameter_map["beta"]: BETA_BEST,
})
unitary = bound.remove_final_measurements(inplace=False)
ideal_probabilities = Statevector.from_instruction(unitary).probabilities()

ideal_rows = []
for index, probability in enumerate(ideal_probabilities):
    bits = [(index >> i) & 1 for i in range(NUM_ASSETS)]
    x = np.asarray(bits, dtype=float)
    ideal_rows.append({
        "bitstring": "".join(map(str, bits)),
        "bits": bits,
        "probability": float(probability),
        "selected_count": int(x.sum()),
        "return": float(RETURNS @ x),
        "risk": float(x @ SIGMA @ x),
        "energy": qubo_energy(bits),
        "feasible": int(x.sum()) == BUDGET,
    })
ideal_df = pd.DataFrame(ideal_rows).sort_values("probability", ascending=False)
display(ideal_df.head(8))

optimal_bits = classical_optimum["bits"]
optimal_probability = float(ideal_probabilities[sum(bit << i for i, bit in enumerate(optimal_bits))])
feasible_probability = float(ideal_df.loc[ideal_df["feasible"], "probability"].sum())
print(f"Ideal probability of exact classical optimum: {optimal_probability:.4%}")
print(f"Ideal probability of any feasible portfolio: {feasible_probability:.4%}")
print(f"Ideal probability of infeasible portfolios: {1-feasible_probability:.4%}")
ideal_df.to_csv(TABLES_DIR / "ideal_qaoa_distribution.csv", index=False)

## 9. Processor A/B reference topologies and transpilation

The guide’s reference architectures are used as supplied. Both processors use the same logical circuit, basis gates, optimization level, and transpiler seed. The physical topologies are comparison models, not claims about real hardware.

In [ ]:
PROCESSOR_A_EDGES = [(0, 1), (1, 2), (1, 3), (3, 4)]
PROCESSOR_B_EDGES = [
    (0, 1), (1, 2), (1, 6), (2, 3), (2, 6), (2, 5),
    (3, 4), (4, 5), (5, 6), (6, 7), (6, 8), (8, 0),
]

coupling_map_A = CouplingMap(couplinglist=PROCESSOR_A_EDGES)
coupling_map_B = CouplingMap(couplinglist=PROCESSOR_B_EDGES)
BASIS_GATES = ["rz", "sx", "x", "cx"]
TRANSPILE_SEED = 42
OPTIMIZATION_LEVEL = 1

transpiled_reference = transpile(
    circuit, basis_gates=BASIS_GATES,
    optimization_level=OPTIMIZATION_LEVEL,
    seed_transpiler=TRANSPILE_SEED,
)
transpiled_A = transpile(
    circuit, coupling_map=coupling_map_A, basis_gates=BASIS_GATES,
    optimization_level=OPTIMIZATION_LEVEL,
    seed_transpiler=TRANSPILE_SEED,
)
transpiled_B = transpile(
    circuit, coupling_map=coupling_map_B, basis_gates=BASIS_GATES,
    optimization_level=OPTIMIZATION_LEVEL,
    seed_transpiler=TRANSPILE_SEED,
)

reference_cx = int(transpiled_reference.count_ops().get("cx", 0))

def transpilation_metrics(name, transpiled):
    ops = transpiled.count_ops()
    cx_count = int(ops.get("cx", 0))
    extra_cx = max(0, cx_count - reference_cx)
    return {
        "Processor": name,
        "Physical qubits in circuit": transpiled.num_qubits,
        "Depth": int(transpiled.depth()),
        "Reference CX": reference_cx,
        "Transpiled CX": cx_count,
        "Additional CX vs reference": extra_cx,
        "Explicit SWAP instructions after basis translation": int(ops.get("swap", 0)),
        "SWAP-equivalent estimate (additional CX / 3)": extra_cx / 3.0,
        "RZ": int(ops.get("rz", 0)),
        "SX": int(ops.get("sx", 0)),
        "X": int(ops.get("x", 0)),
    }

transpilation_df = pd.DataFrame([
    transpilation_metrics("Unconstrained reference", transpiled_reference),
    transpilation_metrics("Processor A", transpiled_A),
    transpilation_metrics("Processor B", transpiled_B),
])
display(transpilation_df)
transpilation_df.to_csv(TABLES_DIR / "transpilation_metrics.csv", index=False)

print("Processor A edges:", PROCESSOR_A_EDGES)
print("Processor B edges:", PROCESSOR_B_EDGES)
print("Note: explicit SWAP instructions are generally decomposed into CX gates by the required basis translation.")
print("The SWAP-equivalent value is an estimate from additional CX gates, not an exact count of routing SWAP operations.")

## 10. Required synthetic noise model

The synthetic model applies single-qubit depolarizing error `p = 0.001` to each one-qubit basis gate (`rz`, `sx`, and `x`), two-qubit depolarizing error `p = 0.005` to `cx`, and symmetric readout error `p = 0.01`. This is a simulator model, not an execution on physical hardware.

In [ ]:
P_1Q = 0.001
P_CX = 0.005
P_READOUT = 0.01

noise_model = NoiseModel()
error_1q = depolarizing_error(P_1Q, 1)
error_cx = depolarizing_error(P_CX, 2)
readout_error = ReadoutError([
    [1 - P_READOUT, P_READOUT],
    [P_READOUT, 1 - P_READOUT],
])
noise_model.add_all_qubit_quantum_error(error_1q, ["rz", "sx", "x"])
noise_model.add_all_qubit_quantum_error(error_cx, ["cx"])
noise_model.add_all_qubit_readout_error(readout_error)

simulator = AerSimulator(noise_model=noise_model, basis_gates=BASIS_GATES)
print("Noise model basis gates:", noise_model.basis_gates)
print("1-qubit depolarizing probability:", P_1Q)
print("CX depolarizing probability:", P_CX)
print("Readout error probability:", P_READOUT)
print("Shots per processor:", SHOTS)

## 11. Noisy 4096-shot Processor A/B benchmark

The optimized ideal parameters are held fixed for both topologies so the comparison changes the topology and resulting transpilation/noise exposure—not the optimizer settings. Accuracy is reported as (1) exact-optimum hit rate and (2) feasible-output rate among sampled bitstrings. The sampled best feasible objective and empirical mean objective are also reported.

In [ ]:
def bind_best_parameters(transpiled_circuit):
    pmap = {p.name: p for p in transpiled_circuit.parameters}
    return transpiled_circuit.assign_parameters({
        pmap["gamma"]: GAMMA_BEST,
        pmap["beta"]: BETA_BEST,
    })


def parse_count_bitstring(count_key, num_assets=NUM_ASSETS):
    # Qiskit count strings display the highest-index classical bit on the left.
    compact = count_key.replace(" ", "")
    if len(compact) < num_assets:
        compact = compact.zfill(num_assets)
    return [int(compact[-1 - i]) for i in range(num_assets)]


def run_noisy_benchmark(name, transpiled_circuit):
    bound_physical = bind_best_parameters(transpiled_circuit)
    result = simulator.run(bound_physical, shots=SHOTS, seed_simulator=SEED).result()
    counts = result.get_counts()
    total = sum(counts.values())
    assert total == SHOTS, f"Expected {SHOTS} shots but got {total}."

    optimum_key = tuple(classical_optimum["bits"])
    optimal_count = 0
    feasible_count = 0
    weighted_energy = 0.0
    best_sampled_feasible = None
    distribution = []

    for key, count in counts.items():
        bits = parse_count_bitstring(key)
        x = np.asarray(bits, dtype=float)
        is_feasible = int(x.sum()) == BUDGET
        energy = qubo_energy(bits)
        probability = count / total
        weighted_energy += probability * energy
        if tuple(bits) == optimum_key:
            optimal_count += count
        if is_feasible:
            feasible_count += count
            row = {
                "bits": bits,
                "count": int(count),
                "probability": probability,
                "return": float(RETURNS @ x),
                "risk": float(x @ SIGMA @ x),
                "objective": direct_objective(bits),
                "energy": energy,
            }
            if best_sampled_feasible is None or row["objective"] < best_sampled_feasible["objective"]:
                best_sampled_feasible = row
        distribution.append({
            "processor": name,
            "count_key": key,
            "bits": bits,
            "count": int(count),
            "probability": probability,
            "selected_count": int(x.sum()),
            "feasible": is_feasible,
            "energy": energy,
        })

    summary = {
        "Processor": name,
        "Shots": total,
        "Optimal portfolio hits": optimal_count,
        "Optimal portfolio hit rate": optimal_count / total,
        "Feasible samples": feasible_count,
        "Feasible-output rate": feasible_count / total,
        "Infeasible-output rate": 1 - feasible_count / total,
        "Empirical mean objective": weighted_energy,
        "Best sampled feasible bits": (best_sampled_feasible["bits"] if best_sampled_feasible else None),
        "Best sampled feasible objective": (best_sampled_feasible["objective"] if best_sampled_feasible else None),
    }
    return summary, distribution

summary_A, distribution_A = run_noisy_benchmark("Processor A", transpiled_A)
summary_B, distribution_B = run_noisy_benchmark("Processor B", transpiled_B)
noisy_summary_df = pd.DataFrame([summary_A, summary_B])
distribution_df = pd.DataFrame(distribution_A + distribution_B)
display(noisy_summary_df)
noisy_summary_df.to_csv(TABLES_DIR / "noisy_processor_comparison.csv", index=False)
distribution_df.to_csv(TABLES_DIR / "noisy_shot_distribution.csv", index=False)

assert noisy_summary_df["Shots"].eq(SHOTS).all()
print("PASS: both processor simulations completed with 4096 shots each.")

## 12. Combined comparison table and figures

The table combines transpilation metrics and noisy sampling quality. Because this is a stochastic simulation, small differences can occur when the seed, shot count, or noise assumptions change.

In [ ]:
processor_metrics_df = transpilation_df[transpilation_df["Processor"].isin(["Processor A", "Processor B"])].copy()
combined_df = processor_metrics_df.merge(noisy_summary_df, on="Processor", how="inner")
display(combined_df)
combined_df.to_csv(TABLES_DIR / "processor_ab_final_comparison.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(combined_df["Processor"], combined_df["Depth"])
axes[0].set_title("Transpiled circuit depth")
axes[0].set_ylabel("Depth")
axes[1].bar(combined_df["Processor"], combined_df["Transpiled CX"])
axes[1].set_title("Transpiled CX gate count")
axes[1].set_ylabel("CX gates")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "processor_ab_circuit_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(combined_df["Processor"], combined_df["Optimal portfolio hit rate"] * 100)
axes[0].set_title("Exact-optimum hit rate")
axes[0].set_ylabel("Percent of shots (%)")
axes[1].bar(combined_df["Processor"], combined_df["Feasible-output rate"] * 100)
axes[1].set_title("Feasible-output rate")
axes[1].set_ylabel("Percent of shots (%)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "processor_ab_noisy_quality.png", dpi=160, bbox_inches="tight")
plt.show()

print("Saved tables to:", TABLES_DIR.resolve())
print("Saved figures to:", FIGURES_DIR.resolve())

## 13. Export reproducibility metadata and final checks

This cell writes the experimental assumptions, noise rates, optimizer parameters, and summary results to JSON for the report. Do not replace actual printed metrics with estimates in the final report.

In [ ]:
metadata = {
    "problem_statement": "Qiskit Fall Fest 2026 Phase 1 PS-O2: Portfolio Optimization",
    "official_inputs": {
        "returns": RETURNS.tolist(),
        "budget": BUDGET,
        "covariance_matrix": SIGMA.tolist(),
    },
    "experimental_assumptions": {
        "risk_aversion_q": RISK_AVERSION,
        "qubo_penalty_lambda": PENALTY,
        "qaoa_depth_p": QAOA_P,
        "parameter_bounds": {"gamma": [0.0, 0.2], "beta": [0.0, float(np.pi)]},
        "optimizer": "SciPy differential_evolution",
        "optimizer_seed": SEED,
    },
    "transpilation": {
        "basis_gates": BASIS_GATES,
        "optimization_level": OPTIMIZATION_LEVEL,
        "seed_transpiler": TRANSPILE_SEED,
    },
    "noise_model": {
        "one_qubit_depolarizing_probability": P_1Q,
        "cx_depolarizing_probability": P_CX,
        "readout_error_probability": P_READOUT,
        "shots_per_processor": SHOTS,
    },
    "classical_optimum": classical_optimum,
    "ideal_qaoa": {
        "gamma": GAMMA_BEST,
        "beta": BETA_BEST,
        "expected_energy": IDEAL_EXPECTED_ENERGY,
        "optimal_bitstring_probability": optimal_probability,
        "feasible_output_probability": feasible_probability,
    },
    "noisy_processor_summary": noisy_summary_df.to_dict(orient="records"),
    "transpilation_summary": transpilation_df.to_dict(orient="records"),
}
with open(TABLES_DIR / "benchmark_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

assert max(qubo_errors) < 1e-9
assert classical_optimum["feasible"]
assert set(noisy_summary_df["Processor"]) == {"Processor A", "Processor B"}
assert noisy_summary_df["Shots"].eq(SHOTS).all()
assert np.isfinite(noisy_summary_df["Optimal portfolio hit rate"]).all()

print("ALL NOTEBOOK CHECKS PASSED")
print("Files created:")
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.as_posix())

## 14. Interpretation and limitations

- The exact classical enumeration is the reference optimum for the chosen `q = 2.0` and `λ = 20.0`; those parameter values are experimental assumptions, not numerical values prescribed by the problem guide.
- The QAOA ansatz is depth one. Its probability of sampling the optimum may be modest; report the measured values rather than claiming quantum advantage.
- Processor A and B are simulated reference coupling maps, not physical QPUs. The noisy benchmark is a synthetic simulation using the rates documented above.
- `Explicit SWAP instructions after basis translation` may be zero because SWAPs are decomposed into the allowed basis. `SWAP-equivalent estimate` is additional CX count divided by three and is only a rough overhead indicator, not an exact routing-stage SWAP count.
- The results are specific to the optimizer bounds, seed, noise model, transpiler settings, and finite shot count. Re-run all cells to regenerate them.
- Before final submission, ensure `README.md`, `report.md`, `requirements.txt`, the processor JSON files, and the saved results agree with the values generated by this notebook.